# Portable bare-name dialect

Every MobilityDB operator is callable in PyMEOS under a single, stable
**portable bare name** (`overlaps`, `contains`, `teq`,
`nearestApproachDistance`, ...). The mapping is the cross-binding source
of truth ([RFC #920](https://github.com/MobilityDB/MobilityDB);
contract: `MobilityDB/MEOS-API` `meta/portable-aliases.json`), so the
*same* names work identically on every binding/engine (PyMEOS, JMEOS,
MEOS.NET, MobilityDuck, MobilitySpark, ...): a user learns one reference
and assumes the rest.

The 29 bare names are **type-agnostic** — one name covers every temporal
type family (temporal, geo, cbuffer, npoint, pose, rgeo). Each bare name
dispatches, by the runtime type of its arguments, to the *exact* function
the operator is backed by, so it is identical to the operator by
construction (nothing is reimplemented).

> Requires the portable dialect (`pymeos.portable`), added on the PyMEOS
> 1.4 line by PR #87. Until that release is published, install PyMEOS
> from the PR branch:
> `pip install "git+https://github.com/MobilityDB/PyMEOS@feat/portable-aliases"`

In [ ]:
from pymeos import (
    pymeos_initialize,
    pymeos_finalize,
    TsTzSpan,
    TFloatSeq,
    TGeomPointSeq,
)
from pymeos.portable import (
    overlaps,
    teq,
    nearestApproachDistance,
)

pymeos_initialize()

## One bare name, any type family

The same callable works regardless of the argument types — it resolves
the matching backing function at call time.

In [ ]:
# Topology on time spans -> backed by overlaps_span_span
s1 = TsTzSpan("[2000-01-01, 2000-01-03]")
s2 = TsTzSpan("[2000-01-02, 2000-01-05]")
print("overlaps(span, span) :", overlaps(s1, s2))

# Temporal comparison on temporal floats -> backed by teq_temporal_temporal
a = TFloatSeq("[1@2000-01-01, 3@2000-01-03]")
b = TFloatSeq("[2@2000-01-01, 2@2000-01-03]")
print("teq(tfloat, tfloat)  :", teq(a, b))

# Spatial distance on temporal points -> backed by the nad_* family
p1 = TGeomPointSeq("[Point(0 0)@2000-01-01, Point(2 2)@2000-01-03]")
p2 = TGeomPointSeq("[Point(1 0)@2000-01-01, Point(3 0)@2000-01-03]")
print("nearestApproachDistance(tgeompoint, tgeompoint) :",
      nearestApproachDistance(p1, p2))

## Identical to the operator by construction

A bare name reuses the operator's own backing function — the same native
call the type-qualified API makes — so results match exactly.

In [ ]:
# Portable bare name vs. the type-qualified API: same backing call.
print("portable       :", overlaps(s1, s2))
print("type-qualified :", s1.overlaps(s2))        # Span.overlaps

print("portable       :", teq(a, b))
print("type-qualified :", a.temporal_equal(b))    # Temporal.temporal_equal

In [ ]:
pymeos_finalize()

The full set of 29 portable bare names is `pymeos.portable.__all__`.